# GRU Evaluation
Evaluates a trained GRU checkpoint on a held-out test set.

**Sections**
1. Next-step MSE — replicates the training objective on the test set
2. Next-*n* MSE — autoregressive prediction *n* steps ahead (sweep + single-n)
3. MSE by context length — per-timestep MSE shows how performance improves with more observations
4. Visual comparison — full autoregressive rollout vs ground truth with 2D scene animation

In [ ]:
import sys

sys.path.insert(0, ".")

import h5py
import json
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import HTML
from tqdm.notebook import tqdm

import helpers.nb_utils as nb_utils
import helpers.nb_viz as nb_viz
from pim.viz import save_animation

import importlib
importlib.reload(nb_viz)

## Config — edit these

In [ ]:
CHECKPOINT_PATH = "../runs/4_dset3_gru_512hidden/best_model.pt"
TEST_H5_PATH = "../datasets/3_fixed_refl_inview_brighter_eval/dataset.h5"
DEVICE = "cuda"  # "cuda" / "mps" / "cpu"
BATCH_SIZE = 512
NUM_WORKERS = 6

# ── Sections 2 & 3 knob ───────────────────────────────────────────────────────
N_STEPS_AHEAD = 5  # prediction horizon; 1 = next-step; sweep in §2 always covers 1..T-1

# ── Sections 2 & 4 knob ───────────────────────────────────────────────────────
N_CONTEXT = 10  # real observations fed as warm-up before autoregressive rollout

# ── Section 4 knobs ───────────────────────────────────────────────────────────
ANIM_INTERVAL = 80  # ms between frames

## Load model and test set

In [ ]:
model, ckpt_info = nb_utils.load_model(CHECKPOINT_PATH, DEVICE)

print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"Epoch      : {ckpt_info['epoch']}")
print(f"Val loss   : {ckpt_info['val_loss']:.6f}")
print(f"Device     : {DEVICE}")

with h5py.File(TEST_H5_PATH, "r") as f:
    n_test = f["obs_intensity"].shape[0]
    T_frames = f["obs_intensity"].shape[1]

test_loader = nb_utils.build_loader(
    TEST_H5_PATH, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS
)

print(f"\nTest samples : {n_test:,}")
print(f"Frames (T)   : {T_frames}")

In [ ]:
# ── GRU training curve ────────────────────────────────────────────────────────
metrics_path = Path(CHECKPOINT_PATH).parent / "metrics.jsonl"
metrics = [json.loads(l) for l in metrics_path.read_text().splitlines() if l.strip()]
epochs = [m["epoch"] for m in metrics]
train_loss = [m["train_loss"] for m in metrics]
val_loss = [m["val_loss"] for m in metrics]
best_epoch = ckpt_info["epoch"]
run_name = Path(CHECKPOINT_PATH).parent.name

fig, ax = plt.subplots(figsize=(9, 4), facecolor=nb_viz._BG_HEX)
nb_viz.style_ax(ax)
ax.plot(epochs, train_loss, color="#0072B2", linewidth=1.6, label="train")
ax.plot(epochs, val_loss, color="#D55E00", linewidth=1.6, label="val")
ax.axvline(
    best_epoch,
    color=nb_viz._TICK_COLOR,
    linewidth=1.0,
    linestyle="--",
    alpha=0.6,
    label=f"best (epoch {best_epoch})",
)
ax.set_yscale("log")
ax.set_xlabel("epoch", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_ylabel("loss (log scale)", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_title(
    f"best val loss: {ckpt_info['val_loss']:.6f}  @  epoch {best_epoch}",
    color=nb_viz._TEXT_COLOR,
    fontsize=11,
)
ax.legend(
    labelcolor=nb_viz._TEXT_COLOR,
    facecolor=nb_viz._BG_HEX,
    edgecolor=nb_viz._TICK_COLOR,
    fontsize=10,
)
fig.suptitle(
    f"MODEL:  {run_name}",
    fontsize=15,
    fontweight="bold",
    color=nb_viz._TEXT_COLOR,
    y=1.02,
)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4), facecolor=nb_viz._BG_HEX)
nb_viz.style_ax(ax)
ax.plot(epochs, train_loss, color="#0072B2", linewidth=1.6, label="train")
ax.plot(epochs, val_loss, color="#D55E00", linewidth=1.6, label="val")
ax.axvline(
    best_epoch,
    color=nb_viz._TICK_COLOR,
    linewidth=1.0,
    linestyle="--",
    alpha=0.6,
    label=f"best (epoch {best_epoch})",
)
ax.set_yscale("log")
ax.set_xscale("log")
ax.set_xlabel("epoch", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_ylabel("loss (log scale)", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_title(
    f"best val loss: {ckpt_info['val_loss']:.6f}  @  epoch {best_epoch}",
    color=nb_viz._TEXT_COLOR,
    fontsize=11,
)
ax.legend(
    labelcolor=nb_viz._TEXT_COLOR,
    facecolor=nb_viz._BG_HEX,
    edgecolor=nb_viz._TICK_COLOR,
    fontsize=10,
)
fig.suptitle(
    f"MODEL:  {run_name}",
    fontsize=15,
    fontweight="bold",
    color=nb_viz._TEXT_COLOR,
    y=1.02,
)
plt.tight_layout()
plt.show()

---
## 1 — Next-step MSE (test set, teacher forcing)
Exact replication of the training objective on the held-out test set.

In [ ]:
per_sample_losses = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="next-step MSE"):
        obs = batch["obs_intensity"].to(DEVICE)  # (B, T, R)
        pred, _ = model(obs)  # (B, T-1, R)
        target = obs[:, 1:, :]  # (B, T-1, R)
        sample_mse = F.mse_loss(pred, target, reduction="none").mean(dim=(1, 2))
        per_sample_losses.append(sample_mse.cpu())

per_sample_losses = torch.cat(per_sample_losses).numpy()
print(f"Next-step MSE  mean : {per_sample_losses.mean():.6f}")
print(f"Next-step MSE  std  : {per_sample_losses.std():.6f}")
print(f"Samples evaluated   : {len(per_sample_losses):,}")

---
## 2 — Next-*n* MSE (autoregressive, *n* steps ahead)
After warming up on `N_CONTEXT` real observations, the model predicts *n* steps autoregressively.
MSE is measured at the *n*-th step only.

In [ ]:
def eval_next_n(model, loader, n_context, n_steps, device):
    """Mean MSE of the prediction n_steps ahead after n_context warm-up steps."""
    losses = []
    with torch.no_grad():
        for batch in loader:
            obs = batch["obs_intensity"].to(device)  # (B, T, R)
            target_t = n_context + n_steps
            if target_t >= obs.shape[1]:
                continue

            h = None
            for t in range(n_context):
                x, h = model.step(obs[:, t, :], h)
            for _ in range(n_steps):
                x, h = model.step(x, h)

            mse = F.mse_loss(x, obs[:, target_t, :], reduction="none").mean(dim=1)
            losses.append(mse.cpu())

    return torch.cat(losses).mean().item() if losses else float("nan")


single_n_mse = eval_next_n(model, test_loader, N_CONTEXT, N_STEPS_AHEAD, DEVICE)
print(
    f"N_CONTEXT={N_CONTEXT}, N_STEPS_AHEAD={N_STEPS_AHEAD}  →  MSE = {single_n_mse:.6f}"
)

In [ ]:
max_n = T_frames - N_CONTEXT - 1
ns = list(range(1, max_n + 1))
mses = [
    eval_next_n(model, test_loader, N_CONTEXT, n, DEVICE)
    for n in tqdm(ns, desc="sweep n_steps_ahead")
]

fig, ax = plt.subplots(figsize=(8, 4), facecolor=nb_viz._BG_HEX)
nb_viz.style_ax(ax)
ax.plot(ns, mses, color="#0072B2", linewidth=1.8)
# ax.axvline(N_STEPS_AHEAD, color="#D55E00", linewidth=1.2, linestyle="--",
#    label=f"N_STEPS_AHEAD={N_STEPS_AHEAD}")
ax.set_xlabel("n steps ahead", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_ylabel("mean MSE", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_title(
    f"Next-n MSE  (warm-up = {N_CONTEXT} obs)", color=nb_viz._TEXT_COLOR, fontsize=12
)
ax.legend(
    labelcolor=nb_viz._TEXT_COLOR,
    facecolor=nb_viz._BG_HEX,
    edgecolor=nb_viz._TICK_COLOR,
    fontsize=9,
)
plt.tight_layout()
plt.show()

---
## 3 — MSE by context length
At each context length *c*, teacher-forces through the first *c* observations, then rolls `N_STEPS_AHEAD` steps autoregressively and measures MSE at that horizon.
Setting `N_STEPS_AHEAD = 1` recovers plain next-step MSE per timestep.

In [ ]:
# For each context length c (1 .. T - N_STEPS_AHEAD):
#   teacher-force through obs[0..c-1], then AR roll N_STEPS_AHEAD steps,
#   measure MSE against obs[c + N_STEPS_AHEAD - 1].
# N_STEPS_AHEAD=1: inner AR loop never runs → identical to teacher-forced next-step curve.

T_max = T_frames - N_STEPS_AHEAD
mse_sum = np.zeros(T_max)
mse_count = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="MSE by context"):
        obs = batch["obs_intensity"].to(DEVICE)  # (B, T, R)
        B = obs.shape[0]
        h = None
        for t in range(T_max):
            # consume one real observation (teacher forcing)
            x, h = model.step(obs[:, t, :], h)
            # AR roll N_STEPS_AHEAD - 1 extra steps from this point
            x_ar, h_ar = x, h
            for _ in range(N_STEPS_AHEAD - 1):
                x_ar, h_ar = model.step(x_ar, h_ar)
            # x_ar predicts obs[t + N_STEPS_AHEAD]
            mse = F.mse_loss(x_ar, obs[:, t + N_STEPS_AHEAD, :], reduction="none").mean(dim=1)
            mse_sum[t] += mse.sum().item()
        mse_count += B

mse_by_t = mse_sum / mse_count
context_lengths = np.arange(1, T_max + 1)

fig, ax = plt.subplots(figsize=(8, 4), facecolor=nb_viz._BG_HEX)
nb_viz.style_ax(ax)
ax.plot(context_lengths, mse_by_t, color="#0072B2", linewidth=1.8)
ax.set_xlabel("context length (observations seen)", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_ylabel("mean MSE", color=nb_viz._TEXT_COLOR, fontsize=11)
ax.set_title(
    f"MSE vs context length  ({N_STEPS_AHEAD}-step prediction)",
    color=nb_viz._TEXT_COLOR,
    fontsize=12,
)
plt.tight_layout()
plt.show()

---
## 4 — Visual comparison: rollout vs ground truth
Warm up on `N_CONTEXT` real observations, then roll out autoregressively to end of sequence.

In [ ]:
VIZ_SAMPLE_IDX = 0  # which test sample to animate
N_SAMPLES_VIZ = 9

for i in range(N_SAMPLES_VIZ):
    idx = VIZ_SAMPLE_IDX + i
    scene, obs_depth, obs_id, obs_intensity = nb_utils.load_sample(TEST_H5_PATH, idx)
    T = obs_intensity.shape[0]
    print(f"Sample {idx}  |  T={T} frames  |  {scene.positions.shape[1]} objects")

    pred_rollout = nb_utils.autoregressive_rollout(
        model, obs_intensity, N_CONTEXT, DEVICE
    )
    print(f"Warm-up: {N_CONTEXT} frames  |  Rollout: {T - N_CONTEXT} frames")

    nb_viz.plot_waterfall_pair(
        obs_depth,
        obs_id,
        obs_intensity,
        scene,
        pred_rollout,
        N_CONTEXT,
        title=f"Sample {idx} — actual vs predicted  (warm-up={N_CONTEXT} frames)",
        dark=True,
    )
    plt.show()

In [ ]:
anim = nb_viz.animate_3panel(
    scene,
    obs_depth,
    obs_id,
    obs_intensity,
    pred_rollout,
    N_CONTEXT,
    interval=ANIM_INTERVAL,
    title=f"Sample {VIZ_SAMPLE_IDX}  |  warm-up={N_CONTEXT} frames  |  orange = rollout start",
    dark=True,
)
plt.close()
HTML(anim.to_jshtml())

In [ ]:
# Optional: save animation
# from pathlib import Path
# Path("outputs").mkdir(exist_ok=True)
# save_animation(anim, f"outputs/eval_sample{VIZ_SAMPLE_IDX}_ctx{N_CONTEXT}.gif", fps=20)